# Binary Choice (using NLS)

This notebook shows a simple example binary choice models, estimating by OLS/NLS.

## Load Packages and Extra Functions

The general NLS functions are from the (local) `FinEcmt_MLEGMM` module.

In [1]:
MyModulePath = joinpath(pwd(),"src")
!in(MyModulePath,LOAD_PATH) && push!(LOAD_PATH,MyModulePath)
using FinEcmt_OLS
using FinEcmt_MLEGMM: NLS
using FinEcmt_ProbitTobit

In [2]:
#=
include(joinpath(pwd(),"src","FinEcmt_OLS.jl"))
include(joinpath(pwd(),"src","FinEcmt_MLEGMM.jl"))
using .FinEcmt_OLS
using .FinEcmt_MLEGMM: NLS
using FinEcmt_ProbitTobit
=#

In [3]:
using DelimitedFiles, Statistics

# Loading Data

In [4]:
yx = readdlm("Data/Recessions.csv",',',skipstart=1)   #start on line 2, column 1
(y,x) = (yx[:,1],yx[:,2:end])
(T,k) = size(x)

(659, 4)

# Linear Probability Model


In [5]:
xNames = ["yield curve slope(t-1)","market return(t-1)","recession(t-1)","c"]

bOls = x\y
(b1,StdErr1,V1,yhat1) = NLS(LpmRegrF,LpmdRegrF,y,x,bOls,0)

printmat(bOls,b1,b1./StdErr1;colNames=["OLS","NLS","t"],rowNames=xNames)

                             OLS       NLS         t
yield curve slope(t-1)    -0.017    -0.017    -3.099
market return(t-1)        -0.549    -0.549    -3.004
recession(t-1)             0.879     0.879    26.377
c                          0.038     0.038     3.424



# Probit


In [6]:
xMean = mean(x,dims=1)           #sample means, for marginal effects

(bProb,StdProb,V3,yhatProb) = NLS(ProbitRegrF,ProbitdRegrF,y,x,bOls,0)
meProb = ProbitMeF(bProb,xMean,[3])

(R2Prob,cTab) = BinaryChoiceR2pred(y.>0.5,yhatProb.>0.5)

printmat(bProb,bProb./StdProb,meProb;colNames=["b","t","marg. eff"],rowNames=xNames,width=14)
printlnPs("R2: ",R2Prob,"\n")

                                   b             t     marg. eff
yield curve slope(t-1)        -0.346        -2.113        -0.020
market return(t-1)            -6.153        -2.160        -0.347
recession(t-1)                 3.802         8.232         0.902
c                             -2.025        -8.858        -0.114

      R2:      0.821          



In [7]:
(bLogit,StdLogit,V4,yhatLogit) = NLS(LogitRegrF,LogitdRegrF,y,x,bOls,0)
(R2Logit,cTab) = BinaryChoiceR2pred(y.>0.5,yhatLogit.>0.5)
meLogit = LogitMeF(bLogit,xMean,[3])

printmat(bLogit,bLogit./StdLogit,meLogit;colNames=["b","t","marg. eff"],rowNames=xNames,width=14)
printlnPs("R2: ",R2Logit,"\n")

                                   b             t     marg. eff
yield curve slope(t-1)        -0.665        -2.122        -0.017
market return(t-1)           -11.099        -1.939        -0.278
recession(t-1)                 6.861         7.491         0.901
c                             -3.673        -7.951        -0.092

      R2:      0.821          

